Tools
Model cab request to call tools that perform tasks such as fetching data from database, searching the web, or running code. Tools are pairing of :

1. A schema , inclusing the name of the tool, a description and argument definitions (oftern a JSON Schema)
2. A function or corroutine

In [4]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [5]:
import truststore   
truststore.inject_into_ssl()

In [6]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    max_completion_tokens=512,
)
response = model.invoke("why do parrots talk")
response

AIMessage(content='Parrots are able to "talk" due to their remarkable ability to mimic sounds. Their vocalization abilities are attributed to the anatomy of their vocal organs, particularly the syrinx, which allows them to produce a wide variety of sounds. This biological feature, combined with their intelligence and social nature, enables them to imitate human speech and other environmental sounds.\n\nHere are a few reasons why parrots might mimic human speech:\n\n1. **Social Interaction**: Parrots are social creatures, and mimicking human speech can be a way to interact with their human companions. They often view humans as part of their flock and try to communicate in a manner similar to how they would in the wild with other parrots.\n\n2. **Attention**: Parrots quickly learn that mimicking human speech can earn them attention from their owners. When they replicate sounds or words, humans often respond positively, reinforcing the behavior.\n\n3. **Environmental Stimulation**: In cap

In [9]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather]) 


In [10]:
response = model_with_tools.invoke("What is the weather in Boston")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 52, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.00027, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00027, 'upstream_inference_prompt_cost': 0.00013, 'upstream_inference_completions_cost': 0.00014}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_e64d293026', 'id': 'gen-1791781748-TnINXh0RlXY1VG5Dzm8x', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a12e83-c17f-7010-b258-5e3aff6d2eff-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bost

## Tools Eexcution Loops

In [12]:
# Steps 1 : Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execute tools and collect Results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
messages.append(final_response.text)

In [13]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 53, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0002725, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0002725, 'upstream_inference_prompt_cost': 0.0001325, 'upstream_inference_completions_cost': 0.00014}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_e64d293026', 'id': 'gen-1791782197-Y610FrRPqfx0vhk3SgXk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12e8a-935a-7da1

## Messages

messages  are the fundamental unit of context for models in langchain. they represent the input and output of the models, carrying both the content and mdetadata needed to represent the state when intracting with an LLM Messages are objects tht contain :

Role - identifies the mssage type (e.g. sytem , user)
Content - Represents the actual content of the message
Metadata - optional fields such as response information , message IDs and token usage

Langchain provides a standard message type that works across all model providers, ensuring consitent behavior regardless of the model being called.

## Use Text Prompts when

You have a single and standalone request
You don't need converstation history
You want minimal code complexity


## Message prompts

Alternatively, you can pass in alist of messages to the model by providing a list of message objects

Message types

    system Message - telll the model how to behave and provide context for intractions
    Human message - Represents user input and intractions with the model
    AI message - response generated by the model, inclusidng text content , tool calls and metadata
    Tool message - represents the outputs of tool calls

In [ ]:
model.invoke("Please tell me what is artificial intelligence")

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Please tell me what is artificial intelligence."),
]

response = model.invoke(messages)
response.content

'Artificial intelligence (AI) is a branch of computer science that aims to create machines capable of performing tasks that typically require human intelligence. These tasks include reasoning, learning, problem-solving, perception, language understanding, and even physical action.\n\nThere are several types of AI, which can be categorized as follows:\n\n1. **Narrow AI (Weak AI):** This type of AI is designed and trained for a specific task or a narrow set of tasks. Examples include voice assistants like Siri or Alexa, recommendation systems, and chatbots.\n\n2. **General AI (Strong AI):** This hypothetical type of AI would have the ability to perform any cognitive task that a human can, with the ability to understand, learn, and apply knowledge across a wide range of domains. General AI does not yet exist.\n\n3. **Superintelligent AI:** This is a speculative future type of AI that would surpass human intelligence in all aspects, including creativity, general wisdom, and problem-solving

## Structured output

Models can be requested to provide their reponse in a format matching a given schema. this is useful for ensuring the output can eb easily parsed and used in subsequent processing. Langchain supports multiple
schema types and methods for enforcing structured output.

Pydantic

Pydantic model provide the richest feature set with field validation, desciptions and nested structures

## Message along with parsed structure

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel) :
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating : float = Field(description="The movie rating out of 10")


In [18]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000213A6EE3ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000213A6F78910>, root_client=<openai.OpenAI object at 0x00000213A6EE3C50>, root_async_client=<openai.AsyncOpenAI object at 0x00000213A6F78690>, model_name='openai/gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', max_tokens=512, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The re

In [20]:
model_with_structure.invoke("Provide details about the movie Drishyam3")

Movie(title='Drishyam 3', year=2023, director='Jeethu Joseph', rating=8.5)

## Nested Structure

In [26]:
from pydantic import BaseModel, Field

class Actor (BaseModel) :
    name : str
    role : str

class MoveDetails (BaseModel) :
    title:str
    year  : int
    cast : list[Actor]
    genres : list[str]
    buget : float | None = Field(None, description = "Budget in millions USD")

model_wtih_structure_MD = model.with_structured_output(MoveDetails)
response = model_wtih_structure_MD.invoke("Provide details about the movie Drishyam3")
response

MoveDetails(title='Drishyam 3', year=2023, cast=[Actor(name='Mohanlal', role='Georgekutty'), Actor(name='Meena', role='Rani George'), Actor(name='Ansiba Hassan', role='Anju George'), Actor(name='Esther Anil', role='Anu George'), Actor(name='Asha Sharath', role='IG Geetha Prabhakar')], genres=['Thriller', 'Drama', 'Mystery'], buget=15.0)

## Typed Dict

TypedDict provides a smipler alternative using python built-in typing, ideal when you don't need rutime validation

In [7]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
	"""A movie with details """
	title: Annotated[str, "The title of the movie"]
	year: Annotated[int, "The release year of the movie"]
	director: Annotated[str, "The director of the movie"]
	rating: Annotated[float, "The rating of the movie out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
response = model_withtypeddict.invoke("Please provide the details of movie pushpa")
response 


{'title': 'Pushpa: The Rise',
 'year': 2021,
 'director': 'Sukumar',
 'rating': 7.6}

## DataClasses

A data class is a class typically cotaining mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [9]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class ContactInfo (BaseModel) :
    """Contact information for an individual"""
    name : str | None = Field(None, description = "Name of the individual")
    email : str | None = Field(None, description = "Email address of the individual")
    phone : str | None = Field(None, description = "Phone number of the individual")

agent = create_agent(
    model = model,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" :[{"role": "user", "content": "Extract contact info from : Jon doe, john@example.com,  (555) 123-4567"}]
})

print(result["structured_response"])

name='Jon Doe' email='john@example.com' phone='(555) 123-4567'


In [10]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo :
    """Contact information for an individual"""
    name : str | None = None
    email : str | None = None
    phone : str | None = None

agent = create_agent(
    model = model,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" :[{"role": "user", "content": "Extract contact info from : Jon doe, john@example.com,  (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='Jon Doe', email='john@example.com', phone='(555) 123-4567')
